# ClinProtGym ESM-C SAE downstream analysis

Use this notebook after embeddings, SAE tasks, benchmarks, and SAE ensembles have already been computed. It reloads saved tables/fitness files, lets you switch annotation schemes, and redraws downstream AUC summaries without rerunning the expensive jobs.

In [ ]:
from pathlib import Path
import subprocess
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

REPO_ROOT = Path.cwd()
OUTPUT_ROOT = REPO_ROOT / "data/clinprotgym_esmc_sae"
PYTHON = Path("/net/dali/home/barton/dhw28/.conda/envs/py312/bin/python")
REFRESH_SCRIPT = REPO_ROOT / "scripts/refresh_clinprotgym_clinvar_auc.py"
CLINVAR_SUMMARY = REPO_ROOT / "data/clin_dms_data/data/raw/clinvar/variant_summary_2026-06.txt.gz"

COUNT_READY_DATASETS = [
    "MV_BRCA1_Findlay_2018",
    "MV_BRCA2_Huang_2025",
    "MV_MSH2_Jia_2020",
    "MV_TP53_Kotler_2018",
]

METHOD_ORDER = [
    "Enrichment ratio baseline",
    "popDMS baseline",
    "LLR baseline",
    "Raw embeddings",
    "Raw SAE",
    "Ensemble SAE model",
]

sys.path.insert(0, str(REPO_ROOT))
from scripts import clinprotgym_esmc_sae_pipeline as pipeline

sns.set_theme(style="whitegrid")

## Refresh annotation-derived AUCs

`auto` uses exact nucleotide-HGVS ClinVar annotations when an `hgvs_nt` mapping exists, then falls back to the existing protein-level ClinVar labels for datasets without nucleotide HGVS. The refresh rewrites metrics and generated plots only; it does not rerun embeddings, SAE training, benchmark jobs, or ensemble jobs.

In [ ]:
def refresh_auc_tables(
    datasets=COUNT_READY_DATASETS,
    annotation_scheme="auto",
    force_rebuild_hgvs_cache=False,
    summarize=True,
    skip_prepare=True,
):
    cmd = [
        str(PYTHON),
        str(REFRESH_SCRIPT),
        "--datasets",
        *datasets,
        "--annotation-scheme",
        annotation_scheme,
        "--clinvar-variant-summary",
        str(CLINVAR_SUMMARY),
    ]
    if skip_prepare:
        cmd.append("--skip-prepare")
    if force_rebuild_hgvs_cache:
        cmd.append("--force-rebuild-hgvs-cache")
    if not summarize:
        cmd.append("--no-summarize")
    print(" ".join(cmd))
    return subprocess.run(cmd, cwd=REPO_ROOT, check=True)

# Set RUN_REFRESH = True when you want this notebook to rewrite AUC tables/plots.
RUN_REFRESH = False
if RUN_REFRESH:
    refresh_auc_tables(annotation_scheme="auto", force_rebuild_hgvs_cache=False, summarize=True)

## Load current downstream tables

In [ ]:
def read_csv_if_exists(path):
    path = Path(path)
    return pd.read_csv(path) if path.is_file() else pd.DataFrame()

method_metrics = read_csv_if_exists(OUTPUT_ROOT / "tables/clinprotgym_method_metrics.csv")
ensemble_metrics = read_csv_if_exists(OUTPUT_ROOT / "tables/clinprotgym_sae_ensemble_metrics.csv")
all_metrics = pd.concat([method_metrics, ensemble_metrics], ignore_index=True, sort=False)
audit = read_csv_if_exists(OUTPUT_ROOT / "tables/clinprotgym_annotation_scheme_audit.csv")
selected_rows = read_csv_if_exists(OUTPUT_ROOT / "tables/clinprotgym_best_method_rows_by_dataset.csv")
star_auc = read_csv_if_exists(OUTPUT_ROOT / "tables/clinprotgym_best_method_auc_by_clinvar_review_stars.csv")

for col in ["auc", "spearman_rho"]:
    if col in all_metrics.columns:
        all_metrics[col] = pd.to_numeric(all_metrics[col], errors="coerce")
if "auc" in all_metrics.columns:
    all_metrics["auc_discrimination"] = all_metrics["auc"].map(pipeline.auc_discrimination_value)

display(audit)
display(all_metrics[[c for c in ["dataset", "method_family", "model_label", "auc", "auc_discrimination", "n_benign", "n_pathogenic", "n_variants", "annotation_scheme"] if c in all_metrics.columns]].head())

## Best method AUC by dataset

In [ ]:
best_auc = pipeline.best_auc_rows_by_dataset_method(all_metrics, METHOD_ORDER)
if not best_auc.empty:
    best_auc = best_auc.copy()
    best_auc["auc_discrimination"] = pd.to_numeric(best_auc["auc_discrimination"], errors="coerce")
    best_auc["method_plot_label"] = best_auc["method_family"].map(pipeline.method_plot_label)
    display(best_auc.sort_values(["dataset", "auc_discrimination"], ascending=[True, False]))

    fig, ax = plt.subplots(figsize=(max(10, 1.2 * best_auc["dataset"].nunique() + 3), 5.8))
    sns.barplot(data=best_auc, x="dataset", y="auc_discrimination", hue="method_plot_label", ax=ax)
    ax.axhline(0.5, color="0.65", linestyle="--", linewidth=1)
    ax.set_ylim(0, 1.02)
    ax.set_xlabel("Dataset")
    ax.set_ylabel("Direction-normalized ClinVar AUC")
    ax.set_title("Best non-layer-0 method per family by dataset")
    ax.tick_params(axis="x", rotation=55)
    for tick in ax.get_xticklabels():
        tick.set_horizontalalignment("right")
    ax.legend(title="Method", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
    sns.despine(ax=ax)
    fig.tight_layout()
else:
    print("No AUC rows loaded yet.")

## ClinVar review-star cutoff plots

In [ ]:
if not star_auc.empty:
    plot_df = star_auc.copy()
    plot_df["auc_discrimination"] = pd.to_numeric(plot_df["auc_discrimination"], errors="coerce")
    plot_df = plot_df[np.isfinite(plot_df["auc_discrimination"])]
    g = sns.catplot(
        data=plot_df,
        x="review_cutoff",
        y="auc_discrimination",
        hue="method_plot_label",
        col="dataset",
        col_wrap=2,
        kind="bar",
        height=4.0,
        aspect=1.45,
        sharey=True,
    )
    g.set_axis_labels("ClinVar review-star cutoff", "Direction-normalized AUC")
    for ax in g.axes.flat:
        ax.axhline(0.5, color="0.65", linestyle="--", linewidth=1)
        ax.set_ylim(0, 1.02)
        ax.tick_params(axis="x", rotation=25)
    g.figure.tight_layout()
else:
    print("No star-cutoff table found. Run refresh_auc_tables(..., summarize=True).")

## Inspect one dataset/method fitness file

In [ ]:
def load_fitness(row):
    path = Path(row["fitness_path"])
    return pd.read_csv(path)

DATASET = "MV_BRCA1_Findlay_2018"
dataset_rows = all_metrics[all_metrics["dataset"].eq(DATASET)].copy()
dataset_rows = dataset_rows.sort_values("auc_discrimination", ascending=False)
display(dataset_rows[[c for c in ["method_family", "model_label", "auc", "auc_discrimination", "n_benign", "n_pathogenic", "n_variants", "annotation_scheme", "fitness_path"] if c in dataset_rows.columns]].head(20))

if not dataset_rows.empty:
    row = dataset_rows.iloc[0]
    fitness = load_fitness(row)
    display(fitness.head())
    binary = fitness[fitness["annotation"].isin(["benign", "pathogenic"])].copy()
    binary["fitness"] = pd.to_numeric(binary["fitness"], errors="coerce")
    fig, ax = plt.subplots(figsize=(7.5, 4.8))
    sns.histplot(data=binary, x="fitness", hue="annotation", bins=35, element="step", stat="count", common_norm=False, ax=ax)
    ax.set_title(f"{DATASET}: {row.get('method_family')} fitness by ClinVar annotation")
    sns.despine(ax=ax)
    fig.tight_layout()

## Inspect or edit HGVS annotation caches

For datasets with exact nucleotide HGVS, the refresh writes a variant map and an annotation cache. Edit the cache, then rerun `refresh_auc_tables(..., force_rebuild_hgvs_cache=False)` so the script uses your edited labels instead of rescanning ClinVar.

In [ ]:
def dataset_paths(dataset):
    root = OUTPUT_ROOT / "datasets" / dataset
    return {
        "dataset_dir": root,
        "tables": root / "tables",
        "figures": root / "figures",
        "sequence_data": root / "sequence_data",
        "hgvs_variant_map": root / "tables" / f"{dataset}_hgvs_variant_map.csv",
        "hgvs_cache": OUTPUT_ROOT / "tables/hgvs_clinvar_annotations" / f"{dataset}_hgvs_clinvar_annotations.csv",
        "method_metrics": root / "tables" / f"{dataset}_method_metrics.csv",
        "ensemble_metrics": root / "tables" / f"{dataset}_sae_ensemble_metrics.csv",
        "sae_metrics": root / "tables" / f"{dataset}_fixed_deltaembsae_layer_metrics.csv",
    }

def load_hgvs_variant_map(dataset):
    return read_csv_if_exists(dataset_paths(dataset)["hgvs_variant_map"])

def load_hgvs_cache(dataset):
    return read_csv_if_exists(dataset_paths(dataset)["hgvs_cache"])

hgvs_map = load_hgvs_variant_map(DATASET)
hgvs_cache = load_hgvs_cache(DATASET)
display(hgvs_map.head())
display(hgvs_cache["annotation"].value_counts(dropna=False) if not hgvs_cache.empty and "annotation" in hgvs_cache.columns else hgvs_cache.head())

## Saved embedding/SAE artifacts

In [ ]:
def list_artifacts(dataset):
    paths = dataset_paths(dataset)
    rows = []
    for label, path in paths.items():
        rows.append({"artifact": label, "path": str(path), "exists": path.exists()})
    emb_status = paths["tables"] / f"{dataset}_embedding_cache_status.csv"
    if emb_status.is_file():
        rows.append({"artifact": "embedding_cache_status", "path": str(emb_status), "exists": True})
    return pd.DataFrame(rows)

display(list_artifacts(DATASET))
display(read_csv_if_exists(dataset_paths(DATASET)["sae_metrics"]).head())